# Model to transform infrastructure

- enables to transform the methane into a hydrogen network
- allows for new investments on defined edges
- bidirectional net flows on edges possible

### Import packages

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import networkx as nx
import os

### Model run definitions

In [2]:
#use shortage nodes to avoid infeasibilities
model_with_shortage = False #True or False

### Import data

In [3]:
# Specify the path to your Excel file
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
excel_file_path = '\Data_full_network.xlsx'  

input_file_path  = input_file_path + excel_file_path
full_input_path = os.path.abspath(os.path.join(os.getcwd(), input_file_path))

# Read the Excel file into a DataFrame
df_nodes = pd.read_excel(full_input_path, sheet_name='Nodes')
df_commodities = pd.read_excel(full_input_path, sheet_name='Commodities')
df_edges = pd.read_excel(full_input_path, sheet_name='Edges')
df_parameter = pd.read_excel(full_input_path, sheet_name='Parameters')
df_supply_values = pd.read_excel(full_input_path, sheet_name='Supply')

### Create input data structure

In [4]:
#commodities
commodities = df_commodities['Commodities'].tolist()

#nodes
nodes = df_nodes['Nodes'].tolist()

#edges
edges_base = list(zip(df_edges['Source'], df_edges['Destination']))

#initial capacities
capacity_base = {tuple(row[['Commodity', 'Source', 'Destination']]): row['initial_capacities'] 
                 for _, row in df_parameter.iterrows()}

#maximal capacities
capacity_limit_base = {tuple(row[['Commodity', 'Source', 'Destination']]): row['max_capacities'] 
                 for _, row in df_parameter.iterrows()}

#variable transportation cost
cost_base = {tuple(row[['Commodity', 'Source', 'Destination']]): row['costs_edge'] 
                 for _, row in df_parameter.iterrows()}

#investment cost new capacity
investment_cost_base = {tuple(row[['Commodity', 'Source', 'Destination']]): row['new_build_cost'] 
                 for _, row in df_parameter.iterrows()}

#investment cost convert capacity
conversion_cost_base = {tuple(row[['Commodity', 'Source', 'Destination']]): row['conversion_cost'] 
                 for _, row in df_parameter.iterrows()}

#converseion capacity factor
conversion_capacity_factor = {tuple(row[['Commodity', 'Source', 'Destination']]): row['conversion_capacity_factor'] 
                 for _, row in df_parameter.iterrows()}

#demand and supply
inflow = {tuple(row[['Commodity', 'Node']]): row['Supply'] 
                 for _, row in df_supply_values.iterrows()}

### adjust input data

In [5]:
#create node types for different model configurations
if model_with_shortage == True:
    #create shortage nodes
    supply_nodes = []
    for (commodity, node), value in inflow.items():
        if value > 0:
            supply_nodes.append(node)
    unique_supply_nodes = list(set(supply_nodes))
    # Add '_shortage' to each element in the list
    unique_supply_nodes_shortage = [node + '_shortage' for node in unique_supply_nodes]
    nodes_shortage = unique_supply_nodes_shortage
    #create new edges for all shortage nodes to the actual node with high-capacity
    # Create a new list by zipping the two lists
    edges_shortage = list(zip(unique_supply_nodes_shortage, unique_supply_nodes))
    # Create a new data structure using the result_list and commodities
    edge_shortage_cap = {}

    for (commodity, suffix, original) in [(c, s, o) for c in commodities for s, o in zip(unique_supply_nodes_shortage, unique_supply_nodes)]:
        edge_shortage_cap[(commodity, suffix, original)] = 10000000

    nodes_network = nodes #+ nodes_shortage
else:
    nodes_network = nodes

In [6]:
# Reverse edges
edges_reverse = [(destination, source) for source, destination in edges_base]

# Combined edges
edges = edges_base + edges_reverse

# Reverse cap
capacity_reverse = {(k, j, i): cap for (k, i, j), cap in capacity_base.items()}

capacity = {**capacity_base, **capacity_reverse}

# Reverse cap
capacity_limit_reverse = {(k, j, i): cap for (k, i, j), cap in capacity_limit_base.items()}

capacity_limit = {**capacity_limit_base, **capacity_limit_reverse}

# Reverse cost
cost_reverse = {(k, j, i): cost for (k, i, j), cost in cost_base.items()}

cost = {**cost_base, **cost_reverse}

# Reverse cost
investment_cost_reverse = {(k, j, i): cost for (k, i, j), cost in investment_cost_base.items()}

investment_cost = {**investment_cost_base, **investment_cost_reverse}

# Reverse cost
conversion_cost_reverse = {(k, j, i): cost for (k, i, j), cost in conversion_cost_base.items()}

conversion_cost = {**conversion_cost_base, **conversion_cost_reverse}


### Network control

In [7]:
#check in all nodes are connected via edges

G = nx.Graph()
G.add_edges_from(edges_base)

# Check if the graph is connected
if nx.is_connected(G):
    print("The graph is connected.")
else:
    print("The graph is not connected.")

The graph is connected.


In [8]:
# Check if all nodes are connected
if set(nodes) in nx.connected_components(G):
    print("All nodes are connected.")
else:
    print("Not all nodes are connected.")

All nodes are connected.


In [13]:
from collections import Counter

In [15]:
if len(edges) != len(set(edges)):
    print("There are duplicates in the list of tuples.")

    # Count occurrences of each edge
    edge_count = Counter(edges)

    # Find duplicates
    duplicates = [edge for edge, count in edge_count.items() if count > 1]

    for dup in duplicates:
        print(dup)
else:
    print("No duplicates found in the list of tuples.")

There are duplicates in the list of tuples.
('node_0110_1', 'node_0110')
('node_0110_2', 'node_0110')
('node_0212_1', 'node_0212')
('node_0212_2', 'node_0212')
('node_0228_1', 'node_0228')
('node_0228_2', 'node_0228')
('node_0234_1', 'node_0234')
('node_0234_2', 'node_0234')
('node_0236_1', 'node_0236')
('node_0236_2', 'node_0236')
('node_0251_1', 'node_0251')
('node_0251_2', 'node_0251')
('node_0260_1', 'node_0260')
('node_0260_2', 'node_0260')
('node_0313_1', 'node_0313')
('node_0313_2', 'node_0313')
('node_0384_1', 'node_0384')
('node_0384_2', 'node_0384')
('node_0395_1', 'node_0395')
('node_0395_2', 'node_0395')
('node_0447_1', 'node_0447')
('node_0447_2', 'node_0447')
('node_0450_1', 'node_0450')
('node_0450_2', 'node_0450')
('node_0482_1', 'node_0482')
('node_0482_2', 'node_0482')
('node_0494_1', 'node_0494')
('node_0494_2', 'node_0494')
('node_0556_1', 'node_0556')
('node_0556_2', 'node_0556')
('node_0110', 'node_0110_1')
('node_0110', 'node_0110_2')
('node_0212', 'node_0212_1')

## Model

### Create model

In [10]:
# Create optimization model
m = gp.Model("Grid_Transformation")

Set parameter Username
Academic license - for non-commercial use only - expires 2024-12-20


### Define variables

In [11]:
# Create variables
if model_with_shortage == True: 
    flow = m.addVars(commodities, edges+edges_shortage, vtype=GRB.CONTINUOUS, name="flow")
else:
    flow = m.addVars(commodities, edges, vtype=GRB.CONTINUOUS, name="flow")
# variable for capacity investment
investment = m.addVars(commodities, edges, vtype=GRB.CONTINUOUS, name="investment")
# Variable for converted capacity
converted_capacity = m.addVars(commodities, edges, vtype=GRB.CONTINUOUS, name="converted_capacity")
# Binary variable to indicate if conversion is active
conversion_indicator = m.addVars(commodities, edges, vtype=GRB.BINARY, name="conversion_indicator")

KeyError: 'Duplicate keys in Model.addVars()'

### Define objective

In [ ]:
# Set objective function
m.setObjective(
    gp.quicksum(flow[h, i, j] * cost[h, i, j] for h in commodities for i, j in edges) +
    gp.quicksum(investment[h, i, j] * investment_cost[h, i, j] for h in commodities for i, j in edges_base) +
    gp.quicksum(converted_capacity[h, i, j] * conversion_cost[h, j, i] for h in commodities for i, j in edges_base),
    GRB.MINIMIZE
)

### Define constraints

In [ ]:
# Edge-capacity constraints with investment
m.addConstrs((flow[h, i, j] <= converted_capacity[h, i, j] + investment[h, i, j] for h in commodities for i, j in edges), "cap")

# Edge-capacity total limit
m.addConstrs((investment[h, i, j] + converted_capacity[h, i, j] 
              <= capacity_limit[h, i, j] for h in commodities for i, j in edges_base), "cap")

# Investment constraints
m.addConstrs((investment[h, i, j] == investment[h, j, i] for h in commodities for i, j in edges), "conversion")

#conversion binary constraint
m.addConstrs((gp.quicksum(conversion_indicator[h, i, j] for h in commodities) == 1 for i, j in edges), "conversion_indicator_sum_one")

#link the converted capacity to the initial capacity
m.addConstrs((capacity[commodities[0], i, j] * conversion_indicator[h, i, j] 
              == converted_capacity[h, i, j] for h in commodities[1:] for i, j in edges), "capacity_relationship")
m.addConstrs((capacity[commodities[0], i, j] * conversion_indicator[commodities[0], i, j]
        == converted_capacity[commodities[0], i, j]
        for i, j in edges), "capacity_relationship_first_commodity")

# Conversion constraints
m.addConstrs((converted_capacity[h, i, j] == converted_capacity[h, j, i] for h in commodities for i, j in edges), "conversion")

# balanced node flows: Kirchhoff's 1st law
m.addConstrs(
    (
        flow.sum(h, "*", j) + inflow[h, j] == flow.sum(h, j, "*")
        for h in commodities
        for j in nodes_network
    ),
    "node",
)

# Flow-conservation constraints
if model_with_shortage == True: 
    m.addConstrs(
        (
            flow.sum(h, "*", j) - flow.sum(h, j, "*") <= 0
            for h in commodities
            for j in nodes_shortage
        ),
        "node",
    )

    # Edge-capacity constraints between shortage nodes and network nodes
    m.addConstrs((flow[h, i, j] <= edge_shortage_cap[h, i, j] for h in commodities for i, j in edges_shortage), "cap")

print('Constraints createt')

### Optimize the model

In [ ]:
# Compute optimal solution
m.optimize()

## Outputs

In [ ]:
if m.Status == GRB.OPTIMAL:
    solution = m.getAttr("X", flow)
    investment_solution = m.getAttr("X", investment)
    converted_capacity_solution = m.getAttr("X", converted_capacity)
    conversion_indicator_solution = m.getAttr("X", conversion_indicator)

    print("****************************")
    print(f"Total cost: {m.objVal}")
    print("****************************")
    
    for h in commodities:
        print(f"\nOptimal flows for {h}:")
        for i, j in edges:
            if solution[h, i, j] > 0:
                print(f"{i} -> {j}: {solution[h, i, j]:g}")

        for i, j in edges_shortage:
            if solution[h, i, j] > 0:
                print(f"{i} -> {j}: {solution[h, i, j]:g}")

        print(f"\nOptimal investments for {h}:")
        for i, j in edges_base:
            if investment_solution[h, i, j] > 0:
                print(f"Investment for {h} in edge ({i} -> {j}): {investment_solution[h, i, j]:g}")

        print(f"\nOptimal converted capacities for {h}:")
        for i, j in edges_base:
            if converted_capacity_solution[h, i, j] > 0:
                print(f"Converted capacity for {h} from {i} to {j}: {converted_capacity_solution[h, i, j]:g}")

    print("\nOptimal conversion indicators:")
    for h in commodities:
        for i, j in edges_base:
            #
                print(f"Conversion of {h} from {i} to {j} is: {conversion_indicator_solution[h, i, j]:g}")
else:
    print("No optimal solution found.")